
# CrowdStrike 2024 Outage — Analysis Notebook

This notebook prepares the dataset and charts behind the Streamlit dashboard.
It focuses on three questions:
1. What happened?
2. What hazards did the outage create across sectors?
3. Why is this a GRC and business-impact event, not just a technical bug?



## Source-backed anchor facts
- CrowdStrike said the defective update was released at **04:09 UTC** and reverted at **05:27 UTC** on July 19, 2024.
- Microsoft estimated **8.5 million Windows devices** were affected, or **less than 1%** of all Windows machines.
- Delta publicly said the outage disrupted **1.3 million customers**, caused about **7,000 flight cancellations** over five days, and cost roughly **$500 million**.


In [ ]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

BASE = Path('..') if Path('..').joinpath('data').exists() else Path('.')
anchors = pd.read_csv(BASE / 'data' / 'incident_anchors.csv')
airlines = pd.read_csv(BASE / 'data' / 'airline_impacts.csv')
sectors = pd.read_csv(BASE / 'data' / 'sector_hazards.csv')
sources = pd.read_csv(BASE / 'data' / 'sources.csv')

anchors.head(), airlines.head(), sectors.head()



## 1) Incident timeline
A short technical release window produced a much longer operational recovery tail.


In [ ]:

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[78], y=['Update window'], orientation='h', base=[4*60+9],
    text=['04:09 UTC release → 05:27 UTC revert'], textposition='inside'
))
fig.update_layout(
    title='CrowdStrike defective update window on July 19, 2024',
    xaxis=dict(title='UTC time', tickmode='array', tickvals=[240,270,300,330], ticktext=['04:00','04:30','05:00','05:30']),
    yaxis_title=''
)
fig.show()



## 2) Airline impact comparison
The airline sector provides one of the clearest public examples of business impact.


In [ ]:

numeric = airlines.dropna(subset=['cancelled_flights']).copy()
fig = px.bar(
    numeric,
    x='airline', y='cancelled_flights', color='period', barmode='group',
    title='Publicly reported airline flight cancellations'
)
fig.show()


In [ ]:

delta_cards = pd.DataFrame({
    'Metric': ['Customers disrupted', 'Flights canceled', 'Claimed impact (USD)'],
    'Value': ['1.3M', '7,000', '$500M']
})
delta_cards



## 3) Cross-sector hazard analysis
This incident created **availability**, **continuity**, and **concentration-risk** hazards across multiple critical sectors.


In [ ]:

fig = px.treemap(
    sectors,
    path=[px.Constant('All sectors'), 'sector'],
    values='severity_score',
    color='severity_score',
    hover_data=['hazard_type', 'examples'],
    title='Relative operational hazard by sector'
)
fig.show()



## 4) GRC interpretation
Use this table directly in your report or presentation.


In [ ]:

grc = pd.DataFrame([
    ['CIS Control 16', 'Secure software lifecycle, validation, and release discipline', 'Defective content reached production; staged deployment safeguards were insufficient', 'Change governance and blast-radius control were not proportionate to risk'],
    ['CIS Control 15', 'Service provider governance and critical-vendor oversight', 'Vendor resilience requirements and deployment guardrails were limited', 'Third-party dependency became a concentration-risk issue'],
    ['CIS Control 17', 'Incident response and recovery readiness', 'Recovery required large-scale manual remediation', 'Technical reversal did not equal business recovery'],
    ['NIST SP 800-161', 'Supply chain risk governance integrated into enterprise risk management', 'Vendor outage resilience was not strong enough at the governance / contractual layer', 'Supply chain risk must be treated as enterprise risk'],
], columns=['Framework / Control', 'What it expects', 'What the incident exposed', 'Board-level implication'])
grc



## 5) Source log
This notebook is intentionally transparent about where the data came from.


In [ ]:

sources
